# E-Commerce Sales & Customer Analytics

## 03 — Sales & Product Analytics

This notebook analyzes sales performance across time, products,
countries, and transactions.

### Business Questions

- How much revenue did the business generate?
- How does revenue change over time?
- Which products generate the most revenue?
- Which products sell the most units?
- Which countries contribute the most revenue?
- How many orders are placed?
- What is the average order value?
- What are the strongest and weakest sales periods?

In [1]:
import pandas as pd
import numpy as np

file_path = "../data/online_retail_II.xlsx"

df_2009 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

df_raw = pd.concat(
    [df_2009, df_2010],
    ignore_index=True
)

df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_raw["invoice"] = df_raw["invoice"].astype(str).str.strip()
df_raw["stockcode"] = df_raw["stockcode"].astype(str).str.strip()
df_raw["description"] = df_raw["description"].astype("string").str.strip()
df_raw["country"] = df_raw["country"].astype("string").str.strip()

df_raw["invoicedate"] = pd.to_datetime(
    df_raw["invoicedate"],
    errors="coerce"
)

df_raw["quantity"] = pd.to_numeric(
    df_raw["quantity"],
    errors="coerce"
)

df_raw["price"] = pd.to_numeric(
    df_raw["price"],
    errors="coerce"
)

df_raw["customer_id"] = pd.to_numeric(
    df_raw["customer_id"],
    errors="coerce"
)

df_clean = df_raw.drop_duplicates().copy()

df_clean["is_cancelled"] = (
    df_clean["invoice"]
    .str.upper()
    .str.startswith("C")
)

df_clean["revenue"] = (
    df_clean["quantity"] * df_clean["price"]
)

df_sales = df_clean[
    (~df_clean["is_cancelled"]) &
    (df_clean["quantity"] > 0) &
    (df_clean["price"] > 0)
].copy()

print("Sales rows:", f"{len(df_sales):,}")

Sales rows: 1,007,913


In [2]:
order_summary = (
    df_sales
    .groupby("invoice")
    .agg(
        order_date=("invoicedate", "min"),
        customer_id=("customer_id", "first"),
        country=("country", "first"),
        order_revenue=("revenue", "sum"),
        items=("quantity", "sum"),
        unique_products=("stockcode", "nunique")
    )
    .reset_index()
)

order_summary.head()

,invoice,order_date,customer_id,country,order_revenue,items,unique_products
0,489434,2009-12-01 07:45:00,13085.0,United Kingdom,505.30,166,8
1,489435,2009-12-01 07:46:00,13085.0,United Kingdom,145.80,60,4
2,489436,2009-12-01 09:06:00,13078.0,United Kingdom,630.33,193,19
3,489437,2009-12-01 09:08:00,15362.0,United Kingdom,310.75,145,23
4,489438,2009-12-01 09:24:00,18102.0,United Kingdom,2286.24,826,17


In [3]:
# ============================================================
# FINAL SALES KPI VALIDATION & EXPORT
# ============================================================

sales_kpis = {
    "Total Revenue": df_sales["revenue"].sum(),
    "Total Orders": df_sales["invoice"].nunique(),
    "Total Units": df_sales["quantity"].sum(),
    "Unique Products": df_sales["stockcode"].nunique(),
    "Unique Customers": df_sales["customer_id"].nunique(),
    "Average Order Value": (
        df_sales["revenue"].sum() / df_sales["invoice"].nunique()
    )
}

print("FINAL SALES KPIs")
print("-" * 40)

for metric, value in sales_kpis.items():
    if metric == "Total Revenue":
        print(f"{metric}: £{value:,.2f}")
    elif metric == "Average Order Value":
        print(f"{metric}: £{value:,.2f}")
    else:
        print(f"{metric}: {value:,.0f}")


# ------------------------------------------------------------
# Final dataset validation
# ------------------------------------------------------------

print("\nFINAL DATASET VALIDATION")
print("-" * 40)

print(f"Rows: {len(df_sales):,}")
print(f"Columns: {df_sales.shape[1]}")
print(f"Date range: {df_sales['invoicedate'].min()} → {df_sales['invoicedate'].max()}")
print(f"Missing Customer IDs: {df_sales['customer_id'].isna().sum():,}")
print(f"Cancelled rows: {df_sales['is_cancelled'].sum():,}")


# ------------------------------------------------------------
# Export final cleaned dataset
# ------------------------------------------------------------

cleaned_path = "../data/cleaned_sales.csv"

df_sales.to_csv(cleaned_path, index=False)

print(f"\nFinal dataset exported to: {cleaned_path}")
print(f"Exported rows: {len(df_sales):,}")

FINAL SALES KPIs
----------------------------------------
Total Revenue: £20,476,260.45
Total Orders: 40,077
Total Units: 11,205,148
Unique Products: 4,916
Unique Customers: 5,878
Average Order Value: £510.92

FINAL DATASET VALIDATION
----------------------------------------
Rows: 1,007,913
Columns: 10
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Missing Customer IDs: 228,488
Cancelled rows: 0

Final dataset exported to: ../data/cleaned_sales.csv
Exported rows: 1,007,913


In [4]:
monthly_sales = (
    df_sales
    .assign(
        year_month=df_sales["invoicedate"].dt.to_period("M")
    )
    .groupby("year_month")
    .agg(
        revenue=("revenue", "sum"),
        orders=("invoice", "nunique"),
        units=("quantity", "sum"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
)

monthly_sales["year_month"] = (
    monthly_sales["year_month"]
    .astype(str)
)

monthly_sales

,year_month,revenue,orders,units,customers
0,2009-12,822483.950,1682,425461,955
1,2010-01,651155.112,1105,390672,720
2,2010-02,551504.726,1201,381879,772
3,2010-03,830915.261,1681,526026,1057
4,2010-04,678875.252,1462,366705,942
5,2010-05,657705.500,1500,395868,966
6,2010-06,749537.310,1645,406820,1041
7,2010-07,648810.270,1529,337895,928
8,2010-08,695251.910,1425,472380,911
9,2010-09,921696.991,1839,583864,1145


In [5]:
product_revenue = (
    df_sales
    .groupby(["stockcode", "description"])
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        orders=("invoice", "nunique"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

product_revenue.head(20)

,stockcode,description,revenue,units_sold,orders,customers
5585,M,Manual,339226.24,9629,784,438
1875,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314
5584,DOT,DOTCOM POSTAGE,309854.11,1415,1415,1
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
3694,47566,PARTY BUNTING,148318.28,28200,2674,894
4939,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860
4612,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010
5587,POST,POSTAGE,125682.42,5363,1851,405
1465,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,896


In [6]:
top_quantity_products = (
    product_revenue
    .sort_values("units_sold", ascending=False)
    .head(20)
)

top_quantity_products

,stockcode,description,revenue,units_sold,orders,customers
4145,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,24445.61,106139,1019,482
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
4612,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010
2790,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138
4939,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860
120,17003,BROCADE RING PURSE,14766.42,70369,456,215
1374,21977,PACK OF 60 PINK PAISLEY CAKE CASES,28081.73,56061,1993,767
4773,84991,60 TEATIME FAIRY CAKE CASES,27041.21,54028,2127,822
1593,22197,SMALL POPCORN HOLDER,42699.89,48561,1407,426


In [7]:
product_revenue["avg_selling_price"] = (
    product_revenue["revenue"] /
    product_revenue["units_sold"]
)

product_revenue.head()

,stockcode,description,revenue,units_sold,orders,customers,avg_selling_price
5585,M,Manual,339226.24,9629,784,438,35.229644
1875,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314,12.485472
5584,DOT,DOTCOM POSTAGE,309854.11,1415,1415,1,218.978170
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490,2.735721
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1,2.080000


In [8]:
country_sales = (
    df_sales
    .groupby("country")
    .agg(
        revenue=("revenue", "sum"),
        orders=("invoice", "nunique"),
        units=("quantity", "sum"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

country_sales.head(20)

,country,revenue,orders,units,customers
40,United Kingdom,1.741020e+07,36535,9187753,5350
11,EIRE,6.587673e+05,626,336329,5
26,Netherlands,5.540381e+05,228,383879,22
15,Germany,4.250197e+05,789,225154,107
14,France,3.504561e+05,622,271901,95
0,Australia,1.692835e+05,95,103759,15
34,Spain,1.083325e+05,154,50307,41
36,Switzerland,1.006856e+05,93,52762,22
35,Sweden,9.186982e+04,105,88633,19
10,Denmark,6.858069e+04,43,237471,12


In [9]:
country_sales["revenue_share_pct"] = (
    country_sales["revenue"] /
    country_sales["revenue"].sum() *
    100
).round(2)

country_sales.head(10)

,country,revenue,orders,units,customers,revenue_share_pct
40,United Kingdom,1.741020e+07,36535,9187753,5350,85.03
11,EIRE,6.587673e+05,626,336329,5,3.22
26,Netherlands,5.540381e+05,228,383879,22,2.71
15,Germany,4.250197e+05,789,225154,107,2.08
14,France,3.504561e+05,622,271901,95,1.71
0,Australia,1.692835e+05,95,103759,15,0.83
34,Spain,1.083325e+05,154,50307,41,0.53
36,Switzerland,1.006856e+05,93,52762,22,0.49
35,Sweden,9.186982e+04,105,88633,19,0.45
10,Denmark,6.858069e+04,43,237471,12,0.33


In [10]:
weekday_sales = (
    df_sales
    .assign(
        day_of_week=df_sales["invoicedate"].dt.day_name()
    )
    .groupby("day_of_week")
    .agg(
        revenue=("revenue", "sum"),
        orders=("invoice", "nunique"),
        units=("quantity", "sum")
    )
    .reset_index()
)

weekday_sales

,day_of_week,revenue,orders,units
0,Friday,3323908.883,6097,1678727
1,Monday,3581594.176,6351,2030843
2,Saturday,9803.050,30,5119
3,Sunday,1799147.282,4835,1027317
4,Thursday,4195212.762,8288,2320416
5,Tuesday,4078149.962,7291,2142960
6,Wednesday,3488444.333,7185,1999766


In [11]:
hourly_sales = (
    df_sales
    .assign(
        hour=df_sales["invoicedate"].dt.hour
    )
    .groupby("hour")
    .agg(
        revenue=("revenue", "sum"),
        orders=("invoice", "nunique"),
        units=("quantity", "sum")
    )
    .reset_index()
)

hourly_sales

,hour,revenue,orders,units
0,6,4.250,1,1
1,7,75765.570,77,44875
2,8,528185.570,998,295728
3,9,1769631.031,2754,917811
4,10,2581131.333,4610,1490303
5,11,2535325.463,4903,1447592
6,12,2850377.671,6427,1665523
7,13,2548467.544,5695,1553637
8,14,2321684.731,5071,1189342
9,15,2394943.792,4566,1242240


In [12]:
best_month = monthly_sales.loc[
    monthly_sales["revenue"].idxmax()
]

print("Best sales month:", best_month["year_month"])
print("Revenue:", f"£{best_month['revenue']:,.2f}")
print("Orders:", f"{best_month['orders']:,}")

Best sales month: 2011-11
Revenue: £1,503,866.78
Orders: 2,769


In [13]:
best_product = product_revenue.iloc[0]

print("Top product:", best_product["description"])
print("Revenue:", f"£{best_product['revenue']:,.2f}")
print("Units sold:", f"{best_product['units_sold']:,}")

Top product: Manual
Revenue: £339,226.24
Units sold: 9,629


In [14]:
management_summary = {
    "Total Revenue": df_sales["revenue"].sum(),
    "Total Orders": df_sales["invoice"].nunique(),
    "Total Units Sold": df_sales["quantity"].sum(),
    "Unique Products": df_sales["stockcode"].nunique(),
    "Unique Customers": df_sales["customer_id"].nunique(),
    "Average Order Value": order_summary["order_revenue"].mean(),
    "Top Product": best_product["description"],
    "Top Product Revenue": best_product["revenue"],
    "Best Sales Month": best_month["year_month"],
    "Best Month Revenue": best_month["revenue"],
    "Top Country": country_sales.iloc[0]["country"],
    "Top Country Revenue": country_sales.iloc[0]["revenue"]
}

management_summary

{'Total Revenue': np.float64(20476260.448),
 'Total Orders': 40077,
 'Total Units Sold': np.int64(11205148),
 'Unique Products': 4916,
 'Unique Customers': 5878,
 'Average Order Value': np.float64(510.92298445492423),
 'Top Product': 'Manual',
 'Top Product Revenue': np.float64(339226.24),
 'Best Sales Month': '2011-11',
 'Best Month Revenue': np.float64(1503866.78),
 'Top Country': 'United Kingdom',
 'Top Country Revenue': np.float64(17410196.117)}

In [15]:
top_50_products = (
    product_revenue
    .sort_values("revenue", ascending=False)
    .head(50)
)

top_50_products[
    [
        "stockcode",
        "description",
        "revenue",
        "units_sold",
        "orders",
        "customers"
    ]
]

,stockcode,description,revenue,units_sold,orders,customers
5585,M,Manual,339226.24,9629,784,438
1875,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314
5584,DOT,DOTCOM POSTAGE,309854.11,1415,1415,1
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
3694,47566,PARTY BUNTING,148318.28,28200,2674,894
4939,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860
4612,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010
5587,POST,POSTAGE,125682.42,5363,1851,405
1465,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,896


In [16]:
special_codes = (
    df_sales[
        df_sales["stockcode"].str.contains(
            r"[^0-9]",
            regex=True,
            na=False
        )
    ]
    .groupby(["stockcode", "description"])
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        orders=("invoice", "nunique")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

special_codes.head(50)

,stockcode,description,revenue,units_sold,orders
1624,M,Manual,339226.24,9629,784
1623,DOT,DOTCOM POSTAGE,309854.11,1415,1415
1167,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
1149,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
1626,POST,POSTAGE,125682.42,5363,1851
1153,85099F,JUMBO BAG STRAWBERRY,67985.77,36860,1830
1152,85099C,JUMBO BAG BAROQUE BLACK WHITE,59057.61,30640,1868
10,15056N,EDWARDIAN PARASOL NATURAL,57975.45,11286,1033
9,15056BL,EDWARDIAN PARASOL BLACK,47267.45,9504,857
624,84078A,SET/4 WHITE RETRO STORAGE CUBES,43844.96,1128,328


In [17]:
non_product_keywords = (
    "POSTAGE",
    "CARRIAGE",
    "CHARGE",
    "FEE",
    "BANK",
    "ADJUST",
    "MANUAL",
    "DISCOUNT"
)

non_product_mask = (
    df_sales["description"]
    .fillna("")
    .str.upper()
    .str.contains("|".join(non_product_keywords), regex=True)
)

print(
    "Potential non-product rows:",
    f"{non_product_mask.sum():,}"
)

print(
    "Potential non-product revenue:",
    f"£{df_sales.loc[non_product_mask, 'revenue'].sum():,.2f}"
)

Potential non-product rows: 14,512
Potential non-product revenue: £1,042,598.69


In [18]:
product_revenue.head(30)[
    [
        "stockcode",
        "description",
        "revenue",
        "units_sold",
        "orders",
        "customers"
    ]
]

,stockcode,description,revenue,units_sold,orders,customers
5585,M,Manual,339226.24,9629,784,438
1875,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314
5584,DOT,DOTCOM POSTAGE,309854.11,1415,1415,1
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
3694,47566,PARTY BUNTING,148318.28,28200,2674,894
4939,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860
4612,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010
5587,POST,POSTAGE,125682.42,5363,1851,405
1465,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,896


In [19]:
special_codes = (
    df_sales[
        df_sales["stockcode"].str.contains(
            r"[^0-9]",
            regex=True,
            na=False
        )
    ]
    .groupby(["stockcode", "description"])
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        orders=("invoice", "nunique")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

special_codes.head(50)

,stockcode,description,revenue,units_sold,orders
1624,M,Manual,339226.24,9629,784
1623,DOT,DOTCOM POSTAGE,309854.11,1415,1415
1167,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
1149,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
1626,POST,POSTAGE,125682.42,5363,1851
1153,85099F,JUMBO BAG STRAWBERRY,67985.77,36860,1830
1152,85099C,JUMBO BAG BAROQUE BLACK WHITE,59057.61,30640,1868
10,15056N,EDWARDIAN PARASOL NATURAL,57975.45,11286,1033
9,15056BL,EDWARDIAN PARASOL BLACK,47267.45,9504,857
624,84078A,SET/4 WHITE RETRO STORAGE CUBES,43844.96,1128,328


In [20]:
non_product_keywords = (
    "POSTAGE",
    "CARRIAGE",
    "CHARGE",
    "FEE",
    "BANK",
    "ADJUST",
    "MANUAL",
    "DISCOUNT"
)

non_product_mask = (
    df_sales["description"]
    .fillna("")
    .str.upper()
    .str.contains("|".join(non_product_keywords), regex=True)
)

print(
    "Potential non-product rows:",
    f"{non_product_mask.sum():,}"
)

print(
    "Potential non-product revenue:",
    f"£{df_sales.loc[non_product_mask, 'revenue'].sum():,.2f}"
)

Potential non-product rows: 14,512
Potential non-product revenue: £1,042,598.69


In [21]:
special_codes.head(50)

,stockcode,description,revenue,units_sold,orders
1624,M,Manual,339226.24,9629,784
1623,DOT,DOTCOM POSTAGE,309854.11,1415,1415
1167,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
1149,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
1626,POST,POSTAGE,125682.42,5363,1851
1153,85099F,JUMBO BAG STRAWBERRY,67985.77,36860,1830
1152,85099C,JUMBO BAG BAROQUE BLACK WHITE,59057.61,30640,1868
10,15056N,EDWARDIAN PARASOL NATURAL,57975.45,11286,1033
9,15056BL,EDWARDIAN PARASOL BLACK,47267.45,9504,857
624,84078A,SET/4 WHITE RETRO STORAGE CUBES,43844.96,1128,328


In [22]:
special_codes["description"].value_counts().head(30)

description
METAL SIGN,CUPCAKE SINGLE HOOK        6
SET OF 4 FAIRY CAKE PLACEMATS         4
Manual                                2
WHITE HANGING HEART T-LIGHT HOLDER    2
JUMBO BAG RED RETROSPOT               2
JUMBO BAG STRAWBERRY                  2
JUMBO  BAG BAROQUE BLACK WHITE        2
EDWARDIAN PARASOL NATURAL             2
EDWARDIAN PARASOL BLACK               2
WOODEN FRAME ANTIQUE WHITE            2
PINK HAPPY BIRTHDAY BUNTING           2
TEA TIME PARTY BUNTING                2
BLUE HAPPY BIRTHDAY BUNTING           2
CREAM CUPID HEARTS COAT HANGER        2
HANGING HEART ZINC T-LIGHT HOLDER     2
SINGLE HEART ZINC T-LIGHT HOLDER      2
DOORMAT BLACK FLOCK                   2
SET OF 3 COLOURED  FLYING DUCKS       2
PINK 3 PIECE MINI DOTS CUTLERY SET    2
RED/WHITE DOTS RUFFLED UMBRELLA       2
RED RETROSPOT UMBRELLA                2
RED 3 PIECE MINI DOTS CUTLERY SET     2
CHILDRENS CUTLERY POLKADOT PINK       2
BLUE 3 PIECE MINI DOTS CUTLERY SET    2
TEA TIME TEA TOWELS         

In [23]:
# ============================================================
# NON-PRODUCT INVESTIGATION
# ============================================================

description_text = (
    df_sales["description"]
    .fillna("")
    .astype(str)
    .str.upper()
)

non_product_mask = description_text.str.contains(
    "|".join(non_product_keywords),
    regex=True,
    na=False
)

print("Potential non-product rows identified:", f"{non_product_mask.sum():,}")
print("Current dataset rows:", f"{len(df_sales):,}")

Potential non-product rows identified: 14,512
Current dataset rows: 1,007,913


In [24]:
print(df_sales.columns.tolist())

['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country', 'is_cancelled', 'revenue']


In [25]:
print(df_sales.shape)

(1007913, 10)


In [26]:
suspicious_descriptions = [
    "MANUAL",
    "POSTAGE",
    "CARRIAGE",
    "BANK CHARGES",
    "AMAZON FEE",
    "ADJUSTMENT"
]

suspicious_mask = (
    df_sales["description"]
    .fillna("")
    .str.upper()
    .isin(suspicious_descriptions)
)

suspicious_transactions = (
    df_sales.loc[
        suspicious_mask,
        [
            "stockcode",
            "description",
            "quantity",
            "price",
            "revenue",
            "is_cancelled"
        ]
    ]
    .groupby(
        ["stockcode", "description", "is_cancelled"]
    )
    .agg(
        rows=("stockcode", "size"),
        revenue=("revenue", "sum"),
        units=("quantity", "sum")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

suspicious_transactions

,stockcode,description,is_cancelled,rows,revenue,units
3,M,Manual,False,846,339226.240,9629
4,POST,POSTAGE,False,1851,125682.420,5363
0,AMAZONFEE,AMAZON FEE,False,3,20467.800,3
2,C2,CARRIAGE,False,267,13426.000,268
1,BANK CHARGES,Bank Charges,False,34,519.241,34
5,m,Manual,False,5,15.050,5


In [27]:
suspicious_transactions[
    [
        "stockcode",
        "description",
        "is_cancelled",
        "rows",
        "revenue",
        "units"
    ]
]

,stockcode,description,is_cancelled,rows,revenue,units
3,M,Manual,False,846,339226.240,9629
4,POST,POSTAGE,False,1851,125682.420,5363
0,AMAZONFEE,AMAZON FEE,False,3,20467.800,3
2,C2,CARRIAGE,False,267,13426.000,268
1,BANK CHARGES,Bank Charges,False,34,519.241,34
5,m,Manual,False,5,15.050,5


In [28]:
non_merchandise_codes = {
    "M",
    "m",
    "POST",
    "AMAZONFEE",
    "C2",
    "BANK CHARGES"
}

merchandise_mask = ~(
    df_sales["stockcode"]
    .astype(str)
    .str.upper()
    .isin(non_merchandise_codes)
)

df_merchandise = df_sales.loc[merchandise_mask].copy()

print("Original rows:", f"{len(df_sales):,}")
print("Merchandise rows:", f"{len(df_merchandise):,}")
print("Removed rows:", f"{len(df_sales) - len(df_merchandise):,}")

print(
    "Merchandise revenue:",
    f"£{df_merchandise['revenue'].sum():,.2f}"
)

Original rows: 1,007,913
Merchandise rows: 1,004,907
Removed rows: 3,006
Merchandise revenue: £19,976,923.70


In [29]:
product_revenue_clean = (
    df_merchandise
    .groupby(["stockcode", "description"])
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        orders=("invoice", "nunique"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

product_revenue_clean.head(20)

,stockcode,description,revenue,units_sold,orders,customers
1875,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314
5581,DOT,DOTCOM POSTAGE,309854.11,1415,1415,1
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
3694,47566,PARTY BUNTING,148318.28,28200,2674,894
4939,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860
4612,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010
1465,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,896
2790,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138
4034,79321,CHILLI LIGHTS,80540.88,15841,1135,304


In [30]:
print("BEFORE removing non-merchandise:")
print(f"Revenue: £{df_sales['revenue'].sum():,.2f}")

print("\nAFTER removing non-merchandise:")
print(f"Revenue: £{df_merchandise['revenue'].sum():,.2f}")

print("\nRevenue excluded:")
print(
    f"£{df_sales['revenue'].sum() - df_merchandise['revenue'].sum():,.2f}"
)

BEFORE removing non-merchandise:
Revenue: £20,476,260.45

AFTER removing non-merchandise:
Revenue: £19,976,923.70

Revenue excluded:
£499,336.75


In [31]:
product_revenue_clean.head(20)

,stockcode,description,revenue,units_sold,orders,customers
1875,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314
5581,DOT,DOTCOM POSTAGE,309854.11,1415,1415,1
4970,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490
3335,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
3694,47566,PARTY BUNTING,148318.28,28200,2674,894
4939,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860
4612,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010
1465,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,896
2790,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138
4034,79321,CHILLI LIGHTS,80540.88,15841,1135,304


In [32]:
print("Rows:", f"{len(df_sales):,}")
print("Columns:", len(df_sales.columns))
print("\nRemaining non-product rows:", non_product_mask.sum())
print("Remaining cancelled rows:", df_sales["is_cancelled"].sum())
print("Missing Customer IDs:", df_sales["customer_id"].isna().sum())
print("Zero prices:", (df_sales["price"] == 0).sum())

Rows: 1,007,913
Columns: 10

Remaining non-product rows: 14512
Remaining cancelled rows: 0
Missing Customer IDs: 228488
Zero prices: 0


In [33]:
current_non_product_mask = (
    df_sales["description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.contains("|".join(non_product_keywords), regex=True)
)

print("Current non-product rows:", current_non_product_mask.sum())
print("Current rows:", f"{len(df_sales):,}")

Current non-product rows: 14512
Current rows: 1,007,913


In [34]:
df_sales = df_sales.loc[~current_non_product_mask].copy()

print("Rows after removing non-products:", f"{len(df_sales):,}")

Rows after removing non-products: 993,401


In [35]:
print("Final rows:", f"{len(df_sales):,}")

final_non_product_mask = (
    df_sales["description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.contains("|".join(non_product_keywords), regex=True)
)

print("Final non-product rows:", final_non_product_mask.sum())
print("Final cancelled rows:", df_sales["is_cancelled"].sum())
print("Final zero-price rows:", (df_sales["price"] == 0).sum())
print("Missing Customer IDs:", df_sales["customer_id"].isna().sum())

Final rows: 993,401
Final non-product rows: 0
Final cancelled rows: 0
Final zero-price rows: 0
Missing Customer IDs: 224593


In [36]:
# ============================================================
# FINAL DATASET VALIDATION
# ============================================================

final_non_product_mask = (
    df_sales["description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.contains("|".join(non_product_keywords), regex=True)
)

# Remove any remaining non-product transactions
df_sales = df_sales.loc[~final_non_product_mask].copy()

# Recalculate validation metrics AFTER filtering
remaining_non_product_mask = (
    df_sales["description"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.contains("|".join(non_product_keywords), regex=True)
)

print("FINAL DATASET VALIDATION")
print("-" * 40)
print(f"Final rows: {len(df_sales):,}")
print(f"Final columns: {df_sales.shape[1]}")
print(f"Remaining non-product rows: {remaining_non_product_mask.sum():,}")
print(f"Final cancelled rows: {df_sales['is_cancelled'].sum():,}")
print(f"Final zero-price rows: {(df_sales['price'] == 0).sum():,}")
print(f"Missing Customer IDs: {df_sales['customer_id'].isna().sum():,}")

FINAL DATASET VALIDATION
----------------------------------------
Final rows: 993,401
Final columns: 10
Remaining non-product rows: 0
Final cancelled rows: 0
Final zero-price rows: 0
Missing Customer IDs: 224,593


## Data Export

The cleaned transaction dataset has been validated and is now ready to be reused for customer-level analysis.

The final dataset contains only valid sales transactions and excludes cancelled transactions, zero-price transactions, and identified non-merchandise transactions.

In [37]:
cleaned_path = "../data/cleaned_sales.csv"

df_sales.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved successfully.")
print("Rows:", f"{len(df_sales):,}")
print("File:", cleaned_path)

Cleaned dataset saved successfully.
Rows: 993,401
File: ../data/cleaned_sales.csv
